In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import requests
import time
from Bio import SeqIO

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.utils import resample
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

In [113]:
labels = pd.read_csv("G-HumanEssential.tsv", sep="\t")

entrez_ids = labels["Gene ID"].astype(str).tolist()

url = "https://mygene.info/v3/gene"

mapping = {}

batch_size = 1000

for start in range(0, len(entrez_ids), batch_size):
    batch = entrez_ids[start:start + batch_size]

    response = requests.post(
        url,
        data={
            "ids": ",".join(batch),
            "scopes": "entrezgene",
            "fields": "entrezgene,ensembl.gene,symbol",
            "species": "human"
        }
    )

    response.raise_for_status()

    results = response.json()

    for result in results:
        if result.get("notfound"):
            continue

        entrez = str(result.get("entrezgene"))

        ensembl_data = result.get("ensembl")

        if isinstance(ensembl_data, dict):
            ensembl = ensembl_data.get("gene")

        elif isinstance(ensembl_data, list):
            ensembl = None

            for item in ensembl_data:
                if isinstance(item, dict) and item.get("gene"):
                    ensembl = item["gene"]
                    break

        else:
            ensembl = None

        if entrez and ensembl:
            mapping[entrez] = ensembl

    print(
        f"Processed {min(start + batch_size, len(entrez_ids)):,} "
        f"/ {len(entrez_ids):,} | "
        f"Mapped: {len(mapping):,}"
    )

    time.sleep(0.2)

print("\nFinished.")
print(f"Total labels: {len(entrez_ids):,}")
print(f"Successfully mapped: {len(mapping):,}")
print(f"Not mapped: {len(entrez_ids) - len(mapping):,}")

Processed 1,000 / 18,528 | Mapped: 992
Processed 2,000 / 18,528 | Mapped: 1,981
Processed 3,000 / 18,528 | Mapped: 2,975
Processed 4,000 / 18,528 | Mapped: 3,969
Processed 5,000 / 18,528 | Mapped: 4,957
Processed 6,000 / 18,528 | Mapped: 5,944
Processed 7,000 / 18,528 | Mapped: 6,932
Processed 8,000 / 18,528 | Mapped: 7,923
Processed 9,000 / 18,528 | Mapped: 8,916
Processed 10,000 / 18,528 | Mapped: 9,911
Processed 11,000 / 18,528 | Mapped: 10,901
Processed 12,000 / 18,528 | Mapped: 11,885
Processed 13,000 / 18,528 | Mapped: 12,879
Processed 14,000 / 18,528 | Mapped: 13,867
Processed 15,000 / 18,528 | Mapped: 14,856
Processed 16,000 / 18,528 | Mapped: 15,848
Processed 17,000 / 18,528 | Mapped: 16,838
Processed 18,000 / 18,528 | Mapped: 17,828
Processed 18,528 / 18,528 | Mapped: 18,352

Finished.
Total labels: 18,528
Successfully mapped: 18,352
Not mapped: 176


In [114]:
mapping_df = pd.DataFrame(
    list(mapping.items()),
    columns=["Gene ID", "Ensembl Gene ID"]
)

mapping_df["Gene ID"] = mapping_df["Gene ID"].astype(str)

labels_clean = labels.copy()
labels_clean["Gene ID"] = labels_clean["Gene ID"].astype(str)

labeled_genes = labels_clean.merge(
    mapping_df,
    on="Gene ID",
    how="inner"
)

print("Labeled genes:", len(labeled_genes))
print("\nClass distribution:")
print(labeled_genes[
    "Essentiality (determined from multiple datasets)"
].value_counts())

Labeled genes: 18351

Class distribution:
Essentiality (determined from multiple datasets)
Non-essential    16841
Essential         1510
Name: count, dtype: int64


In [115]:
fasta_path = "Homo_sapiens.GRCh38.cds.all.fa"

target_ensembl_ids = set(
    labeled_genes["Ensembl Gene ID"].astype(str)
)

matched_sequences = {}
matched_headers = {}

for record in SeqIO.parse(fasta_path, "fasta"):
    header = record.description

    gene_id = None

    for field in header.split():
        if field.startswith("gene:"):
            gene_id = field.split(":", 1)[1].split(".", 1)[0]
            break

    if gene_id in target_ensembl_ids:
        sequence = str(record.seq)

        if (
            gene_id not in matched_sequences
            or len(sequence) > len(matched_sequences[gene_id])
        ):
            matched_sequences[gene_id] = sequence
            matched_headers[gene_id] = header

print(f"Target genes:       {len(target_ensembl_ids):,}")
print(f"Genes found in FASTA: {len(matched_sequences):,}")
print(f"Genes not found:     {len(target_ensembl_ids - matched_sequences.keys()):,}")

Target genes:       18,349
Genes found in FASTA: 18,013
Genes not found:     336


In [116]:
label_column = "Essentiality (determined from multiple datasets)"

sequence_df = pd.DataFrame(
    [
        {
            "Ensembl Gene ID": gene_id,
            "sequence": sequence,
        }
        for gene_id, sequence in matched_sequences.items()
    ]
)

dataset = labeled_genes.merge(
    sequence_df,
    on="Ensembl Gene ID",
    how="inner"
)

dataset["label"] = (
    dataset[label_column] == "Essential"
).astype(int)

dataset = dataset[
    ["Gene ID", "Ensembl Gene ID", "sequence", "label"]
]

print(f"Final dataset size: {len(dataset):,}")

print(dataset["label"].value_counts())


Final dataset size: 18,015
label
0    16506
1     1509
Name: count, dtype: int64


In [117]:
from itertools import product

K = 3

kmers = [
    "".join(kmer)
    for kmer in product("ACGT", repeat=K)
]

kmer_to_idx = {kmer: i for i, kmer in enumerate(kmers)}

print(f"Number of possible {K}-mers:", len(kmers))
print(kmers)

Number of possible 3-mers: 64
['AAA', 'AAC', 'AAG', 'AAT', 'ACA', 'ACC', 'ACG', 'ACT', 'AGA', 'AGC', 'AGG', 'AGT', 'ATA', 'ATC', 'ATG', 'ATT', 'CAA', 'CAC', 'CAG', 'CAT', 'CCA', 'CCC', 'CCG', 'CCT', 'CGA', 'CGC', 'CGG', 'CGT', 'CTA', 'CTC', 'CTG', 'CTT', 'GAA', 'GAC', 'GAG', 'GAT', 'GCA', 'GCC', 'GCG', 'GCT', 'GGA', 'GGC', 'GGG', 'GGT', 'GTA', 'GTC', 'GTG', 'GTT', 'TAA', 'TAC', 'TAG', 'TAT', 'TCA', 'TCC', 'TCG', 'TCT', 'TGA', 'TGC', 'TGG', 'TGT', 'TTA', 'TTC', 'TTG', 'TTT']


In [124]:
def kmer_features(sequence, k=3):
    counts = np.zeros(4 ** k, dtype=np.float32)

    total = len(sequence) - k + 1

    if total <= 0:
        return counts

    for i in range(total):
        kmer = sequence[i:i+k]

        if kmer in kmer_to_idx:
            counts[kmer_to_idx[kmer]] += 1

    counts /= total

    return counts


X_kmer = np.vstack([
    kmer_features(seq, K)
    for seq in tqdm(dataset["sequence"], desc="Creating k-mer features")
])

y = dataset["label"].values

print("X shape:", X_kmer.shape)
print("y shape:", y.shape)

Creating k-mer features: 100%|██████████| 18015/18015 [00:05<00:00, 3307.47it/s]

X shape: (18015, 64)
y shape: (18015,)


In [122]:
X_train, X_test, y_train, y_test = train_test_split(
    X_kmer,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining distribution:")
print(pd.Series(y_train).value_counts())

print("\nTesting distribution:")
print(pd.Series(y_test).value_counts())

Training samples: 14412
Testing samples: 3603

Training distribution:
0    13205
1     1207
Name: count, dtype: int64

Testing distribution:
0    3301
1     302
Name: count, dtype: int64


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

majority_mask = y_train == 0
minority_mask = y_train == 1

X_majority, y_majority = X_train_scaled[majority_mask], y_train[majority_mask]
X_minority, y_minority = X_train_scaled[minority_mask], y_train[minority_mask]

X_minority_up, y_minority_up = resample(
    X_minority,
    y_minority,
    replace=True,
    n_samples=len(y_majority),
    random_state=42,
)

X_train_balanced = np.vstack([X_majority, X_minority_up])
y_train_balanced = np.hstack([y_majority, y_minority_up])

shuffle_idx = np.random.RandomState(42).permutation(len(y_train_balanced))
X_train_balanced = X_train_balanced[shuffle_idx]
y_train_balanced = y_train_balanced[shuffle_idx]

print("Original training distribution:")
print(pd.Series(y_train).value_counts())
print("\nBalanced training distribution:")
print(pd.Series(y_train_balanced).value_counts())

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=1e-3,
    batch_size=64,
    learning_rate_init=1e-3,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=42,
)

mlp.fit(X_train_balanced, y_train_balanced)

print("\nTraining finished.")
print("Epochs run:", mlp.n_iter_)
print("Best validation score:", round(mlp.best_validation_score_, 4))

Original training distribution:
0    13205
1     1207
Name: count, dtype: int64

Balanced training distribution:
0    13205
1    13205
Name: count, dtype: int64

Training finished.
Epochs run: 81
Best validation score: 0.9712


In [131]:
y_prob = mlp.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("Classification report (threshold = 0.5):\n")
print(classification_report(y_test, y_pred, target_names=["Non-essential", "Essential"]))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print(f"\nROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_prob):.4f}")

Classification report (threshold = 0.5):

               precision    recall  f1-score   support

Non-essential       0.93      0.94      0.94      3301
    Essential       0.23      0.18      0.20       302

     accuracy                           0.88      3603
    macro avg       0.58      0.56      0.57      3603
 weighted avg       0.87      0.88      0.87      3603

Confusion matrix:
[[3116  185]
 [ 247   55]]

ROC-AUC: 0.7318
PR-AUC:  0.1963
